# DPO on top of 3B SFT (Kaggle) — EXPERIMENTAL

Teaches the model to *prefer* the correct statement over a hallucinated one, using
our contrastive labels as free preference pairs (chosen = true answer, rejected =
a false answer). Policy starts from the **SFT adapter**; DPO refines it.

⚠️ Multimodal DPO (image in the prompt) via TRL is newer than SFT — expect to
debug the training cell together. Run the **SFT-finish notebook first**; it is the
reliable path. This adds the novel SFT→DPO result on top.

**Before running — upload the SFT adapter as a Kaggle Dataset:**
1. From the SFT-finish run, download `adapter_sft_final.zip`; unzip; upload the
   folder as a Kaggle Dataset (e.g. `sft-adapter-q3b`).
   (For a quick test you can instead point at `checkpoint-600` from the partial SFT.)
2. Add Input → that dataset. Accelerator → **GPU T4 x2** (or T4). Secret `HF_TOKEN`.

Outputs: `/kaggle/working/adapter_dpo_final` + `adapter_dpo_final.zip`.

## 1. Install

In [ ]:
import os, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!pip install -q -U "transformers>=4.49.0" "trl>=0.12.0" "peft>=0.10.0" accelerate bitsandbytes qwen-vl-utils datasets
print('GPUs:', torch.cuda.device_count())

## 2. Config + HF login + find SFT adapter

In [ ]:
import os
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret('HF_TOKEN'))
    print('Logged in via Kaggle Secret.')
except Exception as e:
    print('No HF secret (public data still works):', e)

REPO_ID   = 'QCRI/AynVQA-ArabicNLP26'
VLM_MODEL = 'Qwen/Qwen2.5-VL-3B-Instruct'
MAX_PIXELS = 512 * 28 * 28     # smaller for DPO (two forward passes per step)
N_DPO     = 800                # subset of train for DPO pairs (keep it light)

# find the SFT adapter folder (has adapter_config.json) under /kaggle/input
SFT_ADAPTER = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'adapter_config.json' in files and 'adapter_model.safetensors' in files:
        SFT_ADAPTER = root; break
assert SFT_ADAPTER, 'SFT adapter not found under /kaggle/input — add your dataset.'
print('SFT adapter:', SFT_ADAPTER)

## 3. Download train + images, build preference pairs

In [ ]:
import json, random
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm
from PIL import Image
from datasets import Dataset

jsonl = hf_hub_download(REPO_ID, filename='task1b/train_en.jsonl', repo_type='dataset')
train = [json.loads(l) for l in open(jsonl, encoding='utf-8') if l.strip()][:N_DPO]
need = sorted({r['image'] for r in train})
img_path = {}
for rel in tqdm(need, desc='images'):
    img_path[rel] = hf_hub_download(REPO_ID, filename=rel, repo_type='dataset')

PROMPT_TEXT = (
    'You are a visual fact-checker examining an image from the Arab world.\n'
    'Below are THREE statements about this image. Exactly ONE is grounded (True).\n'
    'Statement 1: {s0}\nStatement 2: {s1}\nStatement 3: {s2}\n'
    'On the VERY FIRST line write ONLY: "Answer: X" where X is 1, 2, or 3.')

random.seed(42)
rows = []
for r in train:
    gold = r['labels'].index(True)
    wrong = random.choice([i for i in (0,1,2) if i != gold])
    rows.append({
        'images': [Image.open(img_path[r['image']]).convert('RGB')],
        'prompt': [{'role':'user','content':[{'type':'image'},
                    {'type':'text','text': PROMPT_TEXT.format(
                        s0=r['statements'][0], s1=r['statements'][1], s2=r['statements'][2])}]}],
        'chosen':   [{'role':'assistant','content':[{'type':'text','text': f'Answer: {gold+1}'}]}],
        'rejected': [{'role':'assistant','content':[{'type':'text','text': f'Answer: {wrong+1}'}]}],
    })
dpo_ds = Dataset.from_list(rows)
print('DPO pairs:', len(dpo_ds))

## 4. Load base + SFT adapter as the policy (across 1-2 T4)

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

N_GPUS = torch.cuda.device_count()
max_mem = {i: '13500MiB' for i in range(N_GPUS)}; max_mem['cpu'] = '8GiB'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
processor = AutoProcessor.from_pretrained(VLM_MODEL, max_pixels=MAX_PIXELS)
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_MODEL, torch_dtype=torch.float16, device_map='auto',
    max_memory=max_mem, quantization_config=bnb)
# policy = base + SFT adapter (trainable). DPOTrainer uses adapter-disabled base as reference.
model = PeftModel.from_pretrained(base, SFT_ADAPTER, is_trainable=True)
model.print_trainable_parameters()

## 5. DPO train (EXPERIMENTAL — most likely to need tweaks)

In [ ]:
from trl import DPOConfig, DPOTrainer

cfg = DPOConfig(
    output_dir='/kaggle/working/dpo_ckpts',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=5e-6,
    beta=0.1,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=10,
    save_strategy='steps', save_steps=200, save_total_limit=2,
    optim='paged_adamw_8bit',
    remove_unused_columns=False,
    report_to='none',
)   # length args (max_length/max_prompt_length) omitted — TRL version rejects them; defaults used
trainer = DPOTrainer(
    model=model, ref_model=None,        # ref = SFT policy with adapter disabled
    args=cfg, train_dataset=dpo_ds,
    processing_class=processor,
)
trainer.train()
print('DPO finished.')

## 6. Save DPO adapter + zip

In [ ]:
import glob, zipfile, os
OUT = '/kaggle/working/adapter_dpo_final'
os.makedirs(OUT, exist_ok=True)
model.save_pretrained(OUT); processor.save_pretrained(OUT)
zp = '/kaggle/working/adapter_dpo_final.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in glob.glob(f'{OUT}/*'):
        zf.write(fp, os.path.basename(fp))
print('Saved:', zp, '-> download, then run the inference notebook to score CI.')